In [ ]:
import numpy as np
import pandas as pd

from datetime import datetime
from matplotlib import pyplot as plt

from das_tools.readers import open_das

import das_tools.processing.hammer_utils as utils
from das_tools.hammer_inversion import (EventData, WaveObservation, 
                                        save_events, pick_ridge)
from das_tools.utils import proj_path

In [2]:
data_dir = proj_path.data_dir()
das_data_dir = proj_path.hammer_das_dir()
das_files = [f for f in das_data_dir.glob("*.h5") if f.is_file()]

data_file = data_dir / "channel_geometry_20260908.npz"
chGeoInfo = np.load(data_file)
ch_xy = chGeoInfo["ch_xy"]
refLat,refLon = chGeoInfo["origin_lat"], chGeoInfo["origin_lon"]

kmin,kmax,Nk = 0.01,2.0,201
k_grid = np.linspace(kmin, kmax, Nk)
wave_names = {0: "FG",
            1: "P",
            2: "SH"}
sigma_by_wave = {"FG": 0.10, "P": 0.10, "SH": 0.10}  # placeholder; step3_invert_fixed sets the sigma used
PLOT_EVENTS = False   # waterfall + FK figure for every wave of every event (~86 figures)
events = []
tap_data = utils.get_all_tap_test_geoinfo_inv_all(data_dir)
ch_change_dt = datetime(2025,4,24,20,24,0)

OUT = proj_path.output_dir("tap_inv_batch")
OUT.mkdir(parents=True, exist_ok=True)
event_name = OUT / "dispersion_events_0908.npz"


In [3]:
for row_idx, row in enumerate(tap_data):
    target_time = row['timestamp']
    target_time_str = target_time.strftime("%Y-%m-%d_%H.%M.00")
    # the channels shifted throughout the experiment
    ch_offset = 0 if target_time < ch_change_dt else -8
    # P- and FG- waves or P-, FG- and SH-waves
    n_waves = 2 if row["t2S"] == 0 else 3
    data_file_path = utils.find_das_file(das_files,target_time_str)
    das_data = open_das(
            data_file_path,
            instrument="onyx",
            engine="h5py",
        )
    Fs = das_data.attrs["fs"]
    ch_spacing = das_data.attrs["dx"]    
    cableDist = das_data["distance"].values
    t0 = pd.Timestamp(das_data["time"].values[0])

    event_waves = {}

    for w in range(n_waves):
        t1,t2,d1,d2,f1,f2 = utils.get_window(row,w)

        # Skip placeholder / invalid windows rather than writing fake f=0 curves.
        if (t2 <= t1) or (d2 <= d1) or (f2 <= f1) or (f2 <= 0):
            print(f"Skipping event {row['eventNo']} wave {wave_names[w]}: invalid window")
            continue

        
        start_idx = np.argmin(np.abs(cableDist-d1))
        end_idx = np.argmin(np.abs(cableDist-d2))
        cur_ch_xy = ch_xy[:,start_idx+ch_offset:(end_idx+1)+ch_offset]
        srcXY = np.asarray(utils.ll_to_xy(row["lon"],row["lat"],
                                        refLat,refLon))
        r = np.linalg.norm(cur_ch_xy-srcXY[:,None],axis=0)

        das_segment = das_data.isel(
                        channel=slice(start_idx, end_idx + 1)
                    )

        das_segment = das_segment.sel(
            time=slice(
                t0 + pd.Timedelta(seconds=t1),
                t0 + pd.Timedelta(seconds=t2),
            )
        )

        x = das_segment["strain"].values
        t = das_segment["time"].values
        XFK, freqs = utils.compute_FK(x,Fs,k_grid,r)

        # Ridge pick inside the physical k band (hammer_utils.tap_test_band_mask).
        wave_name = wave_names[w]
        k_low, k_high = utils.tap_test_band_mask(wave_name, freqs)
        k_obs, f_obs = pick_ridge(XFK, freqs, k_grid, f1, f2, k_low, k_high)

        event_waves[wave_name] = WaveObservation(
            wave_type=wave_name,
            frequency=f_obs,
            k_obs=k_obs,
            sigma_k=sigma_by_wave[wave_name],
            metadata={
                "event_id": row["eventNo"],
                "timestamp": target_time,
                "window": {
                    "t1": t1,
                    "t2": t2,
                    "d1": d1,
                    "d2": d2,
                    "f1": f1,
                    "f2": f2,
                },
                "start_idx": start_idx,
                "end_idx": end_idx,
                "ch_offset": ch_offset,
            }
        )


        if PLOT_EVENTS:
            plt.imshow(x.T, aspect='auto',origin='lower',cmap='bwr',vmin=-.1,vmax=.1,
                extent=[t.min(), t.max(), cableDist[start_idx]/1e3, cableDist[end_idx]/1e3])

            plt.title(f"{w}: Hammer Test ({target_time_str} UTC)")
            plt.colorbar(label="Amplitude")
            plt.xlabel("Time (s)")
            plt.ylabel("Distance (km)")
            plt.show()


            XFK_dB = 10*np.log10(XFK)
            XFK_dB_norm = XFK_dB - np.max(XFK_dB,axis=1)[:,None] # XFK_dB.max()#


            plt.figure()
            plt.imshow(XFK_dB_norm, aspect='auto',origin='lower',cmap='magma',
                    vmin=XFK_dB_norm.max()-20, vmax=XFK_dB_norm.max(),
                    extent=[kmin, kmax, freqs.min(), freqs.max()])
            plt.colorbar(label="Normalized Power (dB)")
            plt.scatter(k_obs,f_obs,color='b',label='ridge pick')
            plt.title(f"{w}: Hammer Test ({target_time_str} UTC)")
            plt.ylabel("Frequency (Hz)")
            plt.xlabel("Wavenumber (rad/m)")
            plt.legend(loc='lower right')
            plt.show()

    # Finished this event
    events.append(
        EventData(
            event_id=row["eventNo"],
            waves=event_waves,
            metadata={
                "timestamp": target_time,
                "lat": row["lat"],
                "lon": row["lon"],
            }
        )
    )

save_events(
    events,
    event_name
)

Skipping event 8 wave P: invalid window
Skipping event 16 wave P: invalid window
Skipping event 17 wave P: invalid window
Skipping event 18 wave P: invalid window
Skipping event 19 wave P: invalid window
Skipping event 20 wave P: invalid window
Skipping event 22 wave P: invalid window
Saved 23 events to:
/Users/junsuj/Desktop/git/das-landfast-ice-inversion2025/outputs/tap_inv_batch/dispersion_events_0908.npz
